# 01 · Synthea patients: who is in our pretend hospital?

**Who this is for:** you, after notebook 00.
**You should already know:** pandas basics (`head`, `groupby`, `value_counts`).

**By the end you will be able to:**
- run Synthea reproducibly and explain each command-line flag
- turn raw medical records into one **profile** per patient (age, sex, illnesses, medicines)
- explain why inpatients are older and sicker than the general population, and how we sample them
- place patients on sites, units, beds and monitors without overfilling any unit
- map the data onto the Azure SQL tables (the hospital's "source of record")

**Outline**
1. Setup
2. What Synthea does
3. Run Synthea
4. Look at the raw records
5. Records → patient profiles
6. Admitting ~300 inpatients
7. Sites, units, beds and devices
8. The Azure SQL schema
9. Exercise
10. Pitfalls and next steps

Code used: `generator/synthea.py`, `generator/hospital.py`, `generator/sql/azure_sql_schema.sql`.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from generator import hospital, synthea


def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for folder in [start, *start.parents]:
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found")


ROOT = find_repo_root()
RUN = synthea.SyntheaRun()  # population=2000, seed=42, reference date 2026-11-01, ages 18-95
WORK_DIR = ROOT / "data" / "synthea" / f"p{RUN.population}_s{RUN.seed}"
AS_OF = pd.Timestamp(RUN.reference_date, tz="UTC")
print(RUN)
print("Output folder:", WORK_DIR.relative_to(ROOT), "(git-ignored)")

## 2. What Synthea does

Synthea simulates people **from birth to the reference date**: childhood illnesses, check-ups, chronic diseases, prescriptions, hospital visits.
It follows published disease statistics, so the records *look* like real medical histories, but nobody in them is real.

We use it for **who the patients are**. Where they are right now (which bed, which monitor) is decided by our own code in section 6.

Here is the exact command our code runs:

In [ ]:
java, jar = synthea.tool_paths()
cmd = synthea.build_command(RUN, java, jar)
print(" ".join(cmd[3:]))  # skip the java/jar paths for readability

| Flag | Meaning |
|---|---|
| `-p 2000` | Generate 2,000 **living** people (some who died along the way are also exported) |
| `-s 42` / `-cs 42` | Random seeds: **same seed = same people**, so the data is reproducible |
| `-r 20261101` | Reference date, the "today" of every medical history. This is where our simulated clock starts. |
| `-a 18-95` | Adults only: our hospital has no children's ward |
| `--exporter.csv.included_files=...` | Only write the 4 tables we use. The full export is ~10x bigger. |
| `Massachusetts` | Synthea models US states. Our fictional sites are in MA cities. |

## 3. Run Synthea

Takes about **2 minutes** the first time. After that the function sees the CSVs already exist and returns immediately (same seed gives the same output, so there's no need to rerun).

In [ ]:
%%time
csv_dir = synthea.run_synthea(RUN, WORK_DIR)
sorted(p.name for p in csv_dir.glob("*.csv"))

## 4. Look at the raw records

In [ ]:
tables = synthea.load_tables(csv_dir)

# see the number of rows and columns in each df from tables
pd.DataFrame(
    {name: [len(df), df.shape[1]] for name, df in tables.items()}, index=["rows", "columns"]
)

In [ ]:
tables["patients"][["Id", "FIRST", "LAST", "BIRTHDATE", "DEATHDATE", "GENDER", "CITY"]].head(3)

Synthea adds digits to names (`Jamel269`) so nobody mistakes them for real people. Still, names and birth dates are treated as **sensitive** later: Unity Catalog masks them for analysts (Plan.md §15).

Conditions are stored as **dated episodes**: a `START` and, if resolved, a `STOP`. The most common ones:

In [ ]:
tables["conditions"]["DESCRIPTION"].value_counts().head(12)

Many entries are social or administrative findings ("Medication review due", "Full-time employment"), not diseases.
That's why we don't use everything. We **pick out only the conditions that matter for vitals and deterioration risk**.

## 5. Records → patient profiles

`build_profiles` produces **one row per living patient** with:
- age (on the reference date) and sex
- a True/False flag per condition we care about, counted only if **active** on the reference date (no STOP yet)
- whether they are currently on a **beta-blocker** (these lower heart rate, which the vitals simulator needs to know)

The condition flags are matched by text patterns:

In [ ]:
synthea.CONDITION_KEYWORDS, synthea.BETA_BLOCKERS

In [ ]:
profiles = synthea.build_profiles(tables, AS_OF)
print(len(profiles), "living patients")
profiles.drop(columns=["first_name", "last_name"]).head()

In [ ]:
flags = [*synthea.CONDITION_KEYWORDS, "on_beta_blocker"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
profiles["age"].plot.hist(bins=26, ax=axes[0], title="Age: general population")
(profiles[flags].mean() * 100).sort_values().plot.barh(ax=axes[1], title="% with active condition")
axes[1].set_xlabel("%")
plt.tight_layout()

**Interpretation:** ages spread fairly evenly across adulthood, because this is the *general* population.
Hypertension and diabetes are common. COPD and heart failure are rare in Synthea (a few percent or less), which matters in the next step.

## 6. Admitting ~300 inpatients

Real hospital wards are full of **older, sicker** people, not a random slice of the population.
So instead of picking 300 people uniformly, each person gets an **admission weight**:

- **age:** weight grows exponentially with age (`exp((age - 50) / 40)`)
- **chronic illness:** each active condition adds to the weight
- **COPD or heart failure:** common reasons for admission, so they get an extra boost
- **children** get weight 0

Let's see the weights for a few example people:

In [ ]:
examples = pd.DataFrame(
    {
        "who": ["30, healthy", "60, healthy", "80, healthy", "80, 3 conditions", "80, COPD"],
        "age": [30, 60, 80, 80, 80],
        "n_conditions": [0, 0, 0, 3, 1],
        "copd": [False, False, False, False, True],
        "heart_failure": False,
    }
)
w = hospital.admission_weights(examples)
examples.assign(relative_chance=(w / w[0]).round(1))[["who", "relative_chance"]]

In [ ]:
inpatients = hospital.admit_inpatients(profiles, n=300, seed=42)

comparison = pd.DataFrame(
    {
        "population_%": profiles[flags].mean() * 100,
        "inpatients_%": inpatients[flags].mean() * 100,
    }
).round(1)
print(
    "Median age: population", profiles["age"].median(), "| inpatients", inpatients["age"].median()
)
comparison

In [ ]:
ax = profiles["age"].plot.hist(
    bins=26, density=True, alpha=0.5, label="population", figsize=(7, 3.5)
)
inpatients["age"].plot.hist(bins=26, density=True, alpha=0.6, label="inpatients", ax=ax)
ax.set_xlabel("age")
ax.legend()
ax.set_title("Inpatients are older than the general population");

**Interpretation:** the inpatients' median age is in the 70s (realistic for medical wards), and chronic illness is far more common than in the population.

Heart failure stays rare (a handful of patients) because Synthea itself generates few cases. That's a known limitation of the source data, noted in the model card.

## 7. Sites, units, beds and devices

Three fictional hospitals, each with three unit types:

In [ ]:
hospital.sites_frame()

Each patient goes to a site (in proportion to its size), then to a unit **by preference**:
COPD patients mostly to *respiratory*, heart patients mostly to *step-down*, everyone else mostly to *general*.
A unit is only chosen if it has a **free bed**, so no unit is ever over capacity.

In [ ]:
occupancy = (
    inpatients.groupby("unit_id")
    .size()
    .rename("patients")
    .to_frame()
    .join(hospital.units_frame().set_index("unit_id")["beds"])
    .assign(occupancy_pct=lambda d: (100 * d["patients"] / d["beds"]).round())
)
occupancy

In [ ]:
inpatients[
    ["encounter_id", "site_id", "unit_id", "bed_id", "device_id", "admit_ts", "age", "copd"]
].head()

Every inpatient now has an **encounter** (this hospital stay), a **bed** and a **monitoring device**.
Vitals messages will only carry the `device_id`. The data platform links them back to the patient through the device assignment, exactly as in a real hospital.

## 8. The Azure SQL schema

In Phase 1 this data is loaded into **Azure SQL**, the hospital's source of record. Databricks copies new and changed rows into Bronze with **Lakeflow Connect** (query-based connector). Every table has a `last_updated` column, so each run fetches only rows that changed since the last run.
The full DDL is in `generator/sql/azure_sql_schema.sql`. Here's how our dataframes map onto it:

| SQL table | Comes from | Notes |
|---|---|---|
| `sites`, `units` | `hospital.sites_frame()`, `hospital.units_frame()` | Change slowly (SCD2 in Silver) |
| `patients` | `profiles` (id, names, birth date, sex) | Names and birth date get masked downstream |
| `conditions`, `medications` | Synthea `conditions` / `medications` CSVs | Full history, with start/stop dates |
| `encounters` | `inpatients` (encounter, site, unit, bed, admit time) | Discharges and transfers update this table during the simulation |
| `device_assignments` | `inpatients` (device ↔ encounter ↔ patient) | How vitals get linked to patients |

In [ ]:
schema = (ROOT / "generator" / "sql" / "azure_sql_schema.sql").read_text()
print(schema[schema.index("CREATE TABLE dbo.encounters") : schema.index("-- Which monitor")])

## 9. Exercise: does the respiratory unit really get the COPD patients?

**Your task:** using `inpatients`, compute for each **unit type** (`general`, `step_down`, `respiratory`):
1. the number of patients
2. the % who have COPD
3. the median age

*Hint:* the unit type is the last part of `unit_id` (e.g. `SITE_A-RESPIRATORY`). Try `.str.split("-").str[-1].str.lower()`.

In [ ]:
# Try it yourself
unit_type = None  # a Series of unit types, one per inpatient

summary = None  # groupby unit_type -> count, % COPD, median age
summary

In [ ]:
# Answer (run after trying)
unit_type = inpatients["unit_id"].str.split("-").str[-1].str.lower()
summary = inpatients.groupby(unit_type).agg(
    patients=("patient_id", "size"),
    copd_pct=("copd", lambda s: round(100 * s.mean(), 1)),
    median_age=("age", "median"),
)
summary

You should see a much higher COPD share on the respiratory unit. Unit preferences work, but they aren't absolute (some COPD patients overflow elsewhere when beds run out).

## 10. Pitfalls and next steps

**Pitfall: counting resolved conditions.** A condition with a `STOP` date before the reference date is history, not current illness. `build_profiles` only counts active ones. Try removing that filter and watch prevalence jump.

**Pitfall: loose text matching.** `"diabetes"` also matches *Prediabetes*. That's why the pattern is `(?<!pre)diabetes` (there's a test for it in `tests/test_synthea.py`).

**Pitfall: running Synthea from the repo root.** It writes an `output/` folder wherever it runs. `run_synthea` always runs it inside `data/` (git-ignored).

**Optional extension:** create `synthea.SyntheaRun(population=500, seed=7)` and a new `WORK_DIR`, then compare the inpatient age and condition mix with seed 42.

**Next notebook:** `02_vitals_simulator_walkthrough`. We give each inpatient a heartbeat: vital signs every 5 minutes that behave like a real body.